# Week 6 — Baselines and Splitting Discipline

## Objetivo do notebook

Este notebook tem como objetivo implementar, de forma prática e reproduzível, as rotinas básicas de avaliação discutidas na Week 6 da disciplina.

Até aqui, o trabalho esteve centrado em compreender o dataset: tipos de variáveis, missing values, distribuições e associações. Nesta etapa, o foco muda. A pergunta deixa de ser apenas **“o que os dados mostram?”** e passa a ser **“como avaliar um modelo de forma metodologicamente correta?”**

Para isso, este notebook organiza um primeiro protocolo de avaliação com quatro componentes centrais:

1. **Divisão dos dados em treino, validação e teste**
   Cada subconjunto terá um papel diferente no processo analítico:
   - **treino** para ajustar regras e modelos;
   - **validação** para comparar alternativas;
   - **teste** para avaliação final.

2. **Construção de baselines simples**
   Antes de usar modelos com features, é necessário criar referências mínimas de desempenho.
   - Em **regressão**, usaremos previsões constantes baseadas na **média** ou na **mediana** de `SalePrice` no treino.
   - Em **classificação**, usaremos uma baseline baseada na **classe mais frequente**.

3. **Cálculo de métricas de avaliação**
   Para regressão, vamos calcular:
   - **MAE** (*Mean Absolute Error*)
   - **RMSE** (*Root Mean Squared Error*)

   Para classificação, vamos calcular:
   - **Accuracy**

4. **Criação de uma tarefa de classificação derivada**
   Como o dataset Ames tem naturalmente um alvo numérico (`SalePrice`), criaremos uma variável categórica derivada, `price_tier`, para permitir também uma tarefa de classificação.

## Ideia central

O objetivo deste notebook **não é ainda encontrar o melhor modelo**, mas sim estabelecer uma disciplina de avaliação:

- separar os dados corretamente;
- evitar leakage;
- criar referências mínimas;
- interpretar métricas com contexto.

Em outras palavras, este notebook é menos sobre “performance” e mais sobre **honestidade metodológica**.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Exemplo: carregar o dataset
# Substitua pelo caminho real do seu ficheiro
df = pd.read_csv('./data/raw/AmesHousing.csv')

# Verificação rápida
print(df.shape)
print(df.head())

(2930, 82)
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0     NaN    NaN          NaN   
1   NaN       Reg          Lvl  ...         0     NaN  MnPrv          NaN   
2   NaN       IR1          Lvl  ...         0     NaN    NaN         Gar2   
3   NaN       Reg          Lvl  ...         0     NaN    NaN          NaN   
4   NaN       IR1          Lvl  ...         0     NaN  MnPrv          NaN   

  Misc Val Mo Sold Yr Sold Sale Type  Sale Condition  SalePrice

## 1. Divisão dos dados

In [2]:
# Primeiro, separar features e alvo de regressão
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# 1) separar treino e temp (validação + teste)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# 2) separar validação e teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (2051, 81) (2051,)
Validation: (439, 81) (439,)
Test: (440, 81) (440,)


## 2. Baselines de regressão

In [3]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Baseline da média aprendida no treino
baseline_mean = y_train.mean()
print("Média de SalePrice no treino:", baseline_mean)

# Criar previsões constantes para o conjunto de teste
y_pred_baseline_test = np.full(shape=len(y_test), fill_value=baseline_mean)

# Calcular métricas
mae_test = mean_absolute_error(y_test, y_pred_baseline_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_baseline_test))

print("Baseline mean predictor on TEST")
print("MAE :", mae_test)
print("RMSE:", rmse_test)

Média de SalePrice no treino: 178641.25012189176
Baseline mean predictor on TEST
MAE : 60845.72728824077
RMSE: 85304.62077399496


In [4]:
baseline_median = y_train.median()
print("Mediana de SalePrice no treino:", baseline_median)

y_pred_baseline_median_test = np.full(shape=len(y_test), fill_value=baseline_median)

mae_test_median = mean_absolute_error(y_test, y_pred_baseline_median_test)
rmse_test_median = np.sqrt(mean_squared_error(y_test, y_pred_baseline_median_test))

print("Baseline median predictor on TEST")
print("MAE :", mae_test_median)
print("RMSE:", rmse_test_median)

Mediana de SalePrice no treino: 160000.0
Baseline median predictor on TEST
MAE : 60104.33181818182
RMSE: 89068.79482992091


## 3. Criar price_tier para classificação

In [5]:
# Criar price_tier a partir de SalePrice
df["price_tier"] = pd.qcut(
    df["SalePrice"],
    q=3,
    labels=["low", "mid", "high"]
)

# Ver distribuição
print(df["price_tier"].value_counts())
print(df[["SalePrice", "price_tier"]].head())

price_tier
low     981
mid     980
high    969
Name: count, dtype: int64
   SalePrice price_tier
0     215000       high
1     105000        low
2     172000        mid
3     244000       high
4     189900        mid


In [6]:
from sklearn.metrics import accuracy_score

# Features para classificação
X_cls = df.drop(columns=["SalePrice", "price_tier"])
y_cls = df["price_tier"]

# Split estratificado para manter proporção de classes
X_train_cls, X_temp_cls, y_train_cls, y_temp_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.30,
    random_state=42,
    stratify=y_cls
)

X_val_cls, X_test_cls, y_val_cls, y_test_cls = train_test_split(
    X_temp_cls,
    y_temp_cls,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_cls
)

print("Train:", X_train_cls.shape, y_train_cls.shape)
print("Validation:", X_val_cls.shape, y_val_cls.shape)
print("Test:", X_test_cls.shape, y_test_cls.shape)

Train: (2051, 81) (2051,)
Validation: (439, 81) (439,)
Test: (440, 81) (440,)


## 4. Baseline de classificação

In [7]:
# Descobrir a classe mais frequente no treino
majority_class = y_train_cls.mode()[0]
print("Classe mais frequente no treino:", majority_class)

# Prever essa classe para todos os pontos do teste
y_pred_majority_test = np.full(shape=len(y_test_cls), fill_value=majority_class)

# Accuracy
acc_test = accuracy_score(y_test_cls, y_pred_majority_test)

print("Majority class baseline on TEST")
print("Accuracy:", acc_test)

Classe mais frequente no treino: low
Majority class baseline on TEST
Accuracy: 0.3340909090909091
